# 第10章实践：完成一次可复现的 Ascend-Xyce Wrapper 部署

## 实践任务

1. 确认 vendored Ascend-GMRES dependency 和 wrapper/adapter 入口。
2. 在课程副本中完成离线构建。
3. 选择 U1 或 U2，记录完整启动命令和环境。
4. 保存 stdout/stderr，并检查三个 solver 的收敛、残差和 CSV 行。
5. 比较 total、assembly、linear solver 与 SpMV；说明结论属于 wrapper benchmark，而非完整 netlist 仿真。
6. 人为给出一个不可写结果目录或未知参数，记录并解释失败层次。

## 环境检查

检查 Ascend NPU、CANN 与正式构建门禁后再完成章末实验。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("说明：本章默认运行 Xyce Adapter benchmark，不宣称完整 upstream Xyce 仿真。")


In [ ]:
%%bash
set -e
cd src/ascend_xyce
ASCEND_XYCE_FETCH_XYCE=0 bash scripts/build.sh
# WARMUP=0：矩阵在重复循环前读取/生成一次；每次 app.run 是 fresh solver adapter/prepare/Device state，warmup 只丢弃额外 solver 调用，不代表 warm cache。
ASCEND_XYCE_WARMUP=0 ASCEND_XYCE_REPEAT=1   bash scripts/run.sh --matrix U1 --csv results/chapter_test.csv   2>&1 | tee results/chapter_test.log
test "$(wc -l < results/chapter_test.csv)" -eq 4
awk -F, 'NR > 1 && $11 != 1 { exit 1 } END { if (NR != 4) exit 1 }' results/chapter_test.csv

## 评价标准

- 依赖、build、launch、input、output 路径完整
- 成功判据同时包含退出码与数值正确性
- 能解释应用总时间和 linear solver 时间差异
- 不把 wrapper 描述成完整 Xyce/netlist/MPI/调度器作业

参考答案见 `answer/10.06_answer.md`。

## 工程实践提交物与完成标准

章测必须基于 `src/ascend_xyce/`，不得只回答概念题。操作链：检查 vendored backend → 离线构建 → 配置 warmup/repeat/matrix → 调用 solve → 检查收敛/残差/误差 → 读取 CSV/log → 分层排错。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“Matrix、Solver、Simulation/Linear time、Iterations、Residual、Converged、Error、日志”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 实验记录与练习

验收必须从源码和运行日志同时证明 solver 调用了 Ascend C kernel；Host-only 结果不通过。

完成后回答：实际后端是什么？reference 与 tolerance 是什么？主要耗时来自计算、通信、传输还是同步？改变一个并发或算法参数后，正确性和性能如何变化？参考答案仅通过本章 `answer/` 链接查阅。


## 四类考核

以下四题中，客观题答案唯一，凭 `src/ascend_xyce/` 源码即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：`xyce_adapter.hpp` 中 `LinearSolverKind` 枚举的取值是（　）

A. `CpuSingle`、`CpuOpenMP16`、`AscendDevice`

B. `CpuSingle`、`CpuOpenMP16`、`AscendAccelerated`

C. `CpuSingle`、`CpuOpenMP16`、`AscendDevice`、`HcclDistributed`

D. `CpuSingle`、`AscendAccelerated`

（2）判断（对/错）：active Device solver 由 CMake 目标 `xyce_npu_solver` 直接编译第 07 章 `dis_gmres` 源码，`XyceLinearSolverAdapter::solve` 调用 `dis_gmres::distributed_gmres`；vendored `backend/gmres/Ascend-GMRES` 只提供 CPU baselines 与 legacy Host prototype 源码，其 `npu_spmv.cpp` 文件名不能作为 active `AscendDevice` 的 NPU 证据。（　）

（3）单选：`xyce_benchmark` 输出的 CSV 结构是（　）

A. 每个被选矩阵 3 个 solver 数据行，整个文件共 3 行且无表头

B. CSV 文件整体只有 1 行表头；每个被选矩阵贡献 3 行 solver 数据

C. CSV 文件整体 1 个 solver 数据行加 1 行表头

D. CSV 文件整体 3 个 solver 数据行加 1 行表头

（4）判断（对/错）：wrapper benchmark 不是完整 upstream Xyce：不解析 netlist、不运行 Xyce executable，也不能得出集群/MPI 调度行为结论。（　）

### 2. 简单题

给出一次完整部署运行：离线构建命令（含 vendored backend 门禁证据）、启动命令（环境变量与矩阵）、CSV 行数与日志保存；证明三个 solver 均收敛（converged、residual、error 字段）。

### 3. 中等题

用 CSV 字段把总时间分解为 assembly、linear solver、SpMV 等阶段并说明计时口径；再人为构造一个失败（不可写结果目录或未知参数），记录错误日志并指出失败发生在哪一层（参数解析/构建/运行/结果写回）。

### 4. 困难题

复现性审计：列出本次运行的依赖与构建门禁证据（vendored backend 被链接的 CMake/日志证据）、CSV/log 的完整路径与校验方法；设计一个不修改仓库的验收脚本步骤（构建→运行→解析 CSV→校验收敛），并说明换机器或换 upstream 版本后哪些结论需要重新验证。
